# Isolating Relevant Data

Nievethan Thanienayagam

100750591

Dataset: https://github.com/hudl/open-data

In [10]:
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [11]:
DATA_DIR = Path("open-data-master/data")
OUTPUT_DIR = Path("data/raw_extracted")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "competitions.json", "r", encoding="utf-8") as f:
    df_comp = pd.DataFrame(json.load(f))

print(f"Total competition-season pairs: {len(df_comp)}")
df_comp[['competition_id', 'season_id', 'competition_name', 'season_name', 'competition_gender']].head(10)

Total competition-season pairs: 80


,competition_id,season_id,competition_name,season_name,competition_gender
0,9,281,1. Bundesliga,2023/2024,male
1,9,27,1. Bundesliga,2015/2016,male
2,1267,107,African Cup of Nations,2023,male
3,16,4,Champions League,2018/2019,male
4,16,1,Champions League,2017/2018,male
5,16,2,Champions League,2016/2017,male
6,16,27,Champions League,2015/2016,male
7,16,26,Champions League,2014/2015,male
8,16,25,Champions League,2013/2014,male
9,16,24,Champions League,2012/2013,male


Since the origin data contains a wide variety of matches from a large range of Eras, leagues, and divisions; matches may alter the desired results of the model. Due to the difference in rules and regulations throughout the years, it would be far more effective to select a time period where the rules remain consistent and similar to today's game.

By mapping every `match_id` to it's competition metadata, we can analyze and extract the all the relevant shot data and fields to produce a lightweight dataset for sharing with the team.

In [12]:
match_metadata = {}
for _, comp in df_comp.iterrows():
    comp_id = comp["competition_id"]
    season_id = comp["season_id"]
    match_file = DATA_DIR / "matches" / str(comp_id) / f"{season_id}.json"
    
    if match_file.exists():
        with open(match_file, "r", encoding="utf-8") as f:
            matches = json.load(f)
            for m in matches:
                match_metadata[m["match_id"]] = {
                    "competition_id": comp_id,
                    "competition_name": comp["competition_name"],
                    "competition_gender": comp["competition_gender"],
                    "season_name": comp["season_name"],
                }

                print(f"Found {len(match_metadata)} total matches across all competitions.")

Found 1 total matches across all competitions.
Found 2 total matches across all competitions.
Found 3 total matches across all competitions.
Found 4 total matches across all competitions.
Found 5 total matches across all competitions.
Found 6 total matches across all competitions.
Found 7 total matches across all competitions.
Found 8 total matches across all competitions.
Found 9 total matches across all competitions.
Found 10 total matches across all competitions.
Found 11 total matches across all competitions.
Found 12 total matches across all competitions.
Found 13 total matches across all competitions.
Found 14 total matches across all competitions.
Found 15 total matches across all competitions.
Found 16 total matches across all competitions.
Found 17 total matches across all competitions.
Found 18 total matches across all competitions.
Found 19 total matches across all competitions.
Found 20 total matches across all competitions.
Found 21 total matches across all competitions.
F

In [13]:
all_shots = []
events_dir = DATA_DIR / "events"

for match_id, meta in tqdm(match_metadata.items(), desc="Extracting shots"):
    event_file = events_dir / f"{match_id}.json"
    if not event_file.exists():
        continue
        
    with open(event_file, "r", encoding="utf-8") as f:
        events = json.load(f)
        
    for ev in events:
        if ev.get("type", {}).get("name") == "Shot":
            ev["match_id"] = match_id
            ev.update(meta)
            all_shots.append(ev)

print(f"Extracted {len(all_shots)} total shots!")

Extracting shots: 100%|██████████| 3961/3961 [02:53<00:00, 22.85it/s]

Extracted 101227 total shots!


In [15]:
df_shots = pd.json_normalize(all_shots, sep="_")

df_shots.to_parquet(OUTPUT_DIR / "statsbomb_all_shots.parquet", index=False)

df_newData = pd.read_parquet(OUTPUT_DIR / "statsbomb_all_shots.parquet")
df_newData.head()

,id,index,period,timestamp,minute,second,possession,location,duration,related_events,...,out,shot_one_on_one,shot_saved_to_post,shot_deflected,shot_open_goal,shot_saved_off_target,off_camera,shot_redirect,shot_follows_dribble,shot_kick_off
0,452ac1fc-6e97-459f-8819-62bc82078292,140,1,00:02:13.090,2,13,6,"[107.0, 53.4]",1.680115,[4ea71661-081c-4dc4-bd6f-76a4c252bb8e],...,None,None,None,None,None,None,None,None,None,None
1,345c34d9-2c2f-4cec-860e-3b3704ebc4f3,303,1,00:05:53.050,5,53,14,"[96.5, 39.9]",0.321155,"[5d113c07-b32c-4906-97d1-65734640a93d, 6b62f2e...",...,None,None,None,None,None,None,None,None,None,None
2,6627cd42-a35b-4e4e-9fd3-05d3269bca36,317,1,00:06:30.028,6,30,15,"[88.2, 45.5]",0.192308,"[25da8991-3559-4fd1-88d5-1d9392599292, 5c1bbd2...",...,None,None,None,None,None,None,None,None,None,None
3,3d1c53e1-4b07-4648-806c-2dc0b2ee5347,424,1,00:10:19.546,10,19,20,"[98.5, 49.2]",0.859342,[36fb6df5-7188-4630-bb4b-4e9e1e4214ee],...,None,None,None,None,None,None,None,None,None,None
4,bcdb8320-5e4f-4c64-910a-f4ff7ace5e8b,580,1,00:14:54.051,14,54,29,"[111.2, 37.8]",0.437028,"[0bb6fdf6-f226-4070-bfba-a53c1b65f467, a871c5f...",...,None,None,None,None,None,None,None,None,None,None


The extracted information is now in a .csv file of a small enough size to commit to github. This file will act as the new original dataset, and our steps can be documented through each phase of this project.